In [13]:
def filter_z_threshold(lidar_df, z_percentile=5):
    """
    Filters out points below the specified percentile of the Z-value distribution.

    Parameters:
        lidar_df (pd.DataFrame): LiDAR DataFrame.
        z_percentile (float): Percentile threshold for Z-values.

    Returns:
        pd.DataFrame: Filtered DataFrame with points above the Z-value threshold.
    """
    z_threshold = np.percentile(lidar_df['z'], z_percentile)
    filtered_df = lidar_df[lidar_df['z'] > z_threshold].reset_index(drop=True)
    return filtered_df


In [16]:
def local_maxima_filter(cloud: np.ndarray, window_factor: float, height_percentile: float) -> np.ndarray:
    """
    Detect local maxima in the point cloud using adaptive parameters based on the data range.

    Parameters:
        cloud (np.ndarray): Numpy array of point cloud data (x, y, z).
        window_factor (float): Factor to determine the window size based on data range.
        height_percentile (float): Percentile to determine the minimum height threshold.

    Returns:
        np.ndarray: Points identified as local maxima.
    """
    assert isinstance(cloud, np.ndarray), f"Cloud needs to be a numpy array, not {type(cloud)}"

    #  height threshold based on the percentile
    z_threshold = np.percentile(cloud[:, 2], height_percentile)
    cloud = cloud[cloud[:, 2] > z_threshold]

    #  dynamic window size based on the range of x, y, and z
    x_range = cloud[:, 0].max() - cloud[:, 0].min()
    y_range = cloud[:, 1].max() - cloud[:, 1].min()
    z_range = cloud[:, 2].max() - cloud[:, 2].min()

    # Dynamic window size based on the Z-range relative to the data
    window_size = window_factor * z_range

    tree = KDTree(data=cloud[:, :2])  # Use only x, y for spatial proximity
    seen_mask = np.zeros(cloud.shape[0], dtype=bool)
    local_maxima = []

    for i, point in enumerate(cloud):
        if seen_mask[i]:
            continue

        neighbor_indices = tree.query_ball_point(point[:2], window_size)
        neighbors = cloud[neighbor_indices]

        # Identify the highest neighbor
        highest_neighbor_index = neighbor_indices[np.argmax(neighbors[:, 2])]

        if i == highest_neighbor_index:  # Current point is the local maximum
            local_maxima.append(point)

        # Mark all neighbors as seen
        seen_mask[neighbor_indices] = True

    return np.array(local_maxima)